# Neural Network per K-Means Cluster

This notebook:
1. Runs K-Means (k=**2**) to segment games into two clusters
2. Trains a **separate 5-layer neural network** for each cluster
3. Each cluster's network uses a **different learning rate and dropout rate**

| Cluster | Learning Rate | Dropout | Rationale |
|---------|--------------|---------|----------|
| 0 | 1e-3 | 0.2 | Established games — moderate regularisation |
| 1 | 5e-4 | 0.3 | Simpler games — slower learning, stronger dropout |

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('Libraries imported.')

## 1. Load & Preprocess Data

In [ ]:
from configs.config import RAW_MERGED_PATH
from src.data.loader import load_merged, validate_merged
from src.data.preprocessor import run_preprocessing_pipeline
from src.features.engineer import prepare_features

df_raw = load_merged(RAW_MERGED_PATH, verbose=False)
validate_merged(df_raw)
df = run_preprocessing_pipeline(df_raw, post_release=True, verbose=False)
data = prepare_features(df, post_release=True, return_pipeline=True)

X_full = data['X_train']
y_full = np.expm1(data['y_train'].values)   # raw copiesSold
feature_cols = data['output_cols']
X_raw = data['X_train_raw'].reset_index(drop=True)

print(f'Dataset: {X_full.shape[0]:,} samples  x  {X_full.shape[1]} features')
print(f'Target (copiesSold): mean={y_full.mean():,.0f}  median={np.median(y_full):,.0f}')

## 2. Clean Infinity / NaN Values

In [ ]:
X_full = np.where(np.isinf(X_full), np.nan, X_full)
col_medians = np.nanmedian(X_full, axis=0)
nan_mask = np.isnan(X_full)
X_full[nan_mask] = np.take(col_medians, np.where(nan_mask)[1])

print(f'Inf remaining : {np.isinf(X_full).sum()}')
print(f'NaN remaining : {np.isnan(X_full).sum()}')

## 3. Apply RFECV Feature Selection (if available)

In [ ]:
import os
if os.path.exists('rfecv_selected_ranking.csv'):
    rfecv_df = pd.read_csv('rfecv_selected_ranking.csv')
    selected = rfecv_df[rfecv_df['selected'] == True]['feature'].tolist()
    selected_idx = [i for i, c in enumerate(feature_cols) if c in selected]
    X_full = X_full[:, selected_idx]
    feature_cols = [feature_cols[i] for i in selected_idx]
    print(f'Using {len(feature_cols)} RFECV-selected features.')
else:
    print(f'Using all {X_full.shape[1]} features.')

## 4. K-Means Clustering (k=2)

In [ ]:
N_CLUSTERS = 2

scaler_km = StandardScaler()
X_scaled_km = scaler_km.fit_transform(X_full)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled_km)

print(f'K-Means fitted  |  Inertia: {kmeans.inertia_:,.0f}\n')
for i in range(N_CLUSTERS):
    n = (cluster_labels == i).sum()
    med = np.median(y_full[cluster_labels == i])
    print(f'  Cluster {i}: {n:>7,} games  ({100*n/len(cluster_labels):.1f}%)  '
          f'median sales: {med:>10,.0f}')

## 5. Neural Network Configurations per Cluster

Each cluster gets a **5-layer MLP** (`512 → 256 → 128 → 64 → 32 → 1`) with different learning rate and dropout:

In [ ]:
from src.models.nn import NeuralNetModel

CLUSTER_NN_CONFIGS = {
    0: dict(
        name='Cluster0_NN',
        hidden_layers=[512, 256, 128, 64, 32],
        lr=1e-3,
        dropout_rate=0.2,
        batch_norm=True,
        batch_size=256,
        max_epochs=150,
        patience=15,
        use_log_target=False,
    ),
    1: dict(
        name='Cluster1_NN',
        hidden_layers=[512, 256, 128, 64, 32],
        lr=5e-4,
        dropout_rate=0.3,
        batch_norm=True,
        batch_size=256,
        max_epochs=150,
        patience=15,
        use_log_target=False,
    ),
}

print(f'  {"Cluster":<10} {"LR":<10} {"Dropout":<10} {"Architecture"}')
print('-' * 65)
for c, cfg in CLUSTER_NN_CONFIGS.items():
    arch = ' → '.join(str(h) for h in cfg['hidden_layers']) + ' → 1'
    print(f'  {c:<10} {cfg["lr"]:<10} {cfg["dropout_rate"]:<10} {arch}')

## 6. Train One Neural Network per Cluster

In [ ]:
trained_models = {}
cluster_data_splits = {}
results = []

for cluster_id in range(N_CLUSTERS):
    print(f'\n{"="*60}')
    print(f'CLUSTER {cluster_id}')
    print(f'{"="*60}')

    mask = cluster_labels == cluster_id
    X_c  = X_full[mask]
    y_c  = y_full[mask]
    print(f'Samples : {len(X_c):,}')

    # 70 / 15 / 15 split within each cluster
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X_c, y_c, test_size=0.30, random_state=42)
    X_val, X_tst, y_val, y_tst = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=42)

    cluster_data_splits[cluster_id] = {
        'X_train': X_tr, 'y_train': y_tr,
        'X_val':   X_val, 'y_val':   y_val,
        'X_test':  X_tst, 'y_test':  y_tst,
    }
    print(f'Train={len(X_tr):,}  Val={len(X_val):,}  Test={len(X_tst):,}')

    cfg   = CLUSTER_NN_CONFIGS[cluster_id]
    model = NeuralNetModel(**cfg)
    print(f'lr={cfg["lr"]}  dropout={cfg["dropout_rate"]}')

    model.fit(X_tr, y_tr, X_val=X_val, y_val=y_val, verbose=True)

    val_metrics  = model.evaluate(X_val, y_val, split_name='val')
    test_metrics = model.evaluate(X_tst, y_tst, split_name='test')

    trained_models[cluster_id] = model
    results.append({
        'Cluster':     cluster_id,
        'N_train':     len(X_tr),
        'LR':          cfg['lr'],
        'Dropout':     cfg['dropout_rate'],
        'Val_RMSE':    round(val_metrics['rmse_raw'],  0),
        'Val_MAE':     round(val_metrics['mae_raw'],   0),
        'Val_R2':      round(val_metrics['r2_log'],    4),
        'Test_RMSE':   round(test_metrics['rmse_raw'], 0),
        'Test_MAE':    round(test_metrics['mae_raw'],  0),
        'Test_R2':     round(test_metrics['r2_log'],   4),
    })

print('\n✓ Both cluster models trained.')

## 7. Results Comparison Table

In [ ]:
results_df = pd.DataFrame(results).set_index('Cluster')
print('Per-Cluster Neural Network Performance (raw copiesSold scale):')
print('=' * 65)
display(results_df.style
    .background_gradient(cmap='RdYlGn_r', subset=['Val_RMSE', 'Test_RMSE'])
    .background_gradient(cmap='RdYlGn',   subset=['Val_R2',   'Test_R2'])
    .format({
        'Val_RMSE':  '{:,.0f}',
        'Val_MAE':   '{:,.0f}',
        'Val_R2':    '{:.4f}',
        'Test_RMSE': '{:,.0f}',
        'Test_MAE':  '{:,.0f}',
        'Test_R2':   '{:.4f}',
        'LR':        '{:.0e}',
    })
)

## 8. Training Loss Curves per Cluster

In [ ]:
colors = ['#2196F3', '#4CAF50']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for cluster_id, model in trained_models.items():
    ax  = axes[cluster_id]
    cfg = CLUSTER_NN_CONFIGS[cluster_id]
    train_loss = model.history.get('train_loss', [])
    val_loss   = model.history.get('val_loss',   [])

    ax.plot(train_loss, color=colors[cluster_id], linewidth=2, label='Train')
    if val_loss:
        ax.plot(val_loss, color=colors[cluster_id], linewidth=2,
                linestyle='--', alpha=0.7, label='Val')

    ax.set_title(
        f'Cluster {cluster_id}\nlr={cfg["lr"]}  dropout={cfg["dropout_rate"]}',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss (scaled)')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('Training & Validation Loss — Per-Cluster Neural Networks',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Predicted vs Actual per Cluster

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for cluster_id, model in trained_models.items():
    ax     = axes[cluster_id]
    splits = cluster_data_splits[cluster_id]
    y_true = splits['y_test']
    y_pred = model.predict(splits['X_test'])

    cap  = np.percentile(y_true, 99)
    mask = y_true <= cap

    ax.scatter(y_true[mask], y_pred[mask],
               alpha=0.2, s=6, color=colors[cluster_id])
    ax.plot([0, cap], [0, cap], 'r--', linewidth=1.5, label='Perfect')
    ax.set_xlabel('Actual copiesSold', fontsize=10)
    ax.set_ylabel('Predicted copiesSold', fontsize=10)

    r2  = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    ax.set_title(
        f'Cluster {cluster_id}  R²={r2:.3f}  MAE={mae:,.0f}\n(capped at 99th pct)',
        fontsize=11, fontweight='bold'
    )
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Predicted vs Actual — Per-Cluster NNs (Test Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Export Predictions to CSV

In [ ]:
import os
os.makedirs('cluster_outputs', exist_ok=True)

all_preds = []
for cluster_id, model in trained_models.items():
    mask   = cluster_labels == cluster_id
    X_c    = X_full[mask]
    y_c    = y_full[mask]
    y_pred = model.predict(X_c)

    pred_df = pd.DataFrame({
        'cluster':          cluster_id,
        'actual_copies':    y_c,
        'predicted_copies': y_pred,
        'residual':         y_pred - y_c,
    })
    path = f'cluster_outputs/cluster_{cluster_id}_nn_predictions.csv'
    pred_df.to_csv(path, index=False)
    all_preds.append(pred_df)
    print(f'Cluster {cluster_id} → {path}  ({len(pred_df):,} rows)')

combined = pd.concat(all_preds, ignore_index=True)
combined.to_csv('cluster_outputs/all_clusters_nn_predictions.csv', index=False)
print(f'\nCombined → cluster_outputs/all_clusters_nn_predictions.csv  ({len(combined):,} rows)')